<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-07-blob-processing-classification-routing/notebooks/Ophelie_Page_Level_Metadata_Boundaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Extract Page-Level Metadata and Determine Document Boundaries

Applies the page-level classification logic from the training module to `pharma-blob-sample.pdf`,
a 10-page multi-document blob.

**What this notebook does**

1. Extracts page text with PyPDF2.
2. Runs the **baseline** `is_same_document()` and `classify_document_type()` prompts exactly as written
   in the module.
3. Runs an **improved** version of the same two prompts.
4. Prints both results side by side so any disagreement is visible, and emits the final JSON.

Running the baseline first is deliberate: the last question on the task asks whether the prompt logic
had to change for this file, and that is only answerable if the original version actually ran.

**Before running:** Runtime -> Change runtime type -> T4 or CPU is fine (no local models here).
`GOOGLE_API_KEY` must be set in the Colab Secrets panel with notebook access enabled.

## Step 1: Install and extract page-level content

In [ ]:
!pip install -q PyPDF2 gdown
!pip install -q -U google-genai

In [ ]:
import os
from PyPDF2 import PdfReader

PDF_PATH = "/content/pharma-blob-sample.pdf"
DRIVE_FILE_ID = "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw"  # pharma-blob-sample.pdf

if not os.path.exists(PDF_PATH):
    !gdown "https://drive.google.com/uc?id={DRIVE_FILE_ID}" -O {PDF_PATH} || true

reader = PdfReader(PDF_PATH)
pages = [page.extract_text() for page in reader.pages]
doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]

print(f"{len(doc_pages)} pages extracted\n")
for p in doc_pages:
    first_line = (p["text"] or "").strip().split("\n")[0][:70]
    print(f"page {p['page_num']:>2} | {len(p['text'] or ''):>5} chars | {first_line}")

10 pages extracted

page  0 |  1238 chars | Cytiva
page  1 |   958 chars | Certificate of Quality
page  2 |   960 chars | Certificate of Quality
page  3 |   723 chars | Packaging Component Specification
page  4 |   742 chars | Packaging Component Specification (continued)
page  5 |  1089 chars | Declaration Regarding Transmissible Spongiform
page  6 |  1049 chars | Material Description Sheet
page  7 |   946 chars | Supplier Qualification Record
page  8 |   986 chars | Supplier Qualification Record (continued)
page  9 |   992 chars | Cytiva


In [ ]:
# Full text of every page, so the boundaries can be checked by eye rather than taken on trust.
for p in doc_pages:
    print("=" * 78)
    print(f"PAGE {p['page_num']}")
    print("=" * 78)
    print((p["text"] or "")[:1200])
    print()

PAGE 0
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits
as well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to
brittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to
+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Description Part Number Operating Temperature
High Flow Kit F, Modified, AKTA ready 29477427 +2°C to +40°C
High Flow Gradient C, Modified, AKTA ready 29184612 +2°C to +40°C
Operating Instructions access link

## Step 2: Gemini setup

Uses the `google-genai` SDK directly. temperature=0 because this is a classification task and the
same page should not get a different label on a re-run.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
GEMINI_MODEL = "models/gemini-flash-lite-latest"

CALLS = {"n": 0}

def gemini_model(prompt, max_tokens=40):
    """Single entry point for every LLM call, so both versions are compared on the same model."""
    CALLS["n"] += 1
    import time; time.sleep(5.0)  # free-tier limit is 15 requests/min, so throttle to ~12
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=max_tokens),
    )
    cand = resp.candidates[0]
    parts = (cand.content.parts if cand.content else None) or []
    return "".join((p.text or "") for p in parts)

DOC_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other",
]

print(gemini_model("Reply with the single word: ready"))

ready


## Step 3: Baseline logic, exactly as given in the module

In [ ]:
def is_same_document_baseline(prev_text, curr_text, doc_type=None):
    prompt = f"""
You are checking whether two pages belong to the same document.
Previous page type: {doc_type or 'unknown'}

Previous Page:
{prev_text}

Current Page:
{curr_text}

Respond with only: Yes or No. Do not explain or add commentary.
"""
    response = gemini_model(prompt)
    return response.strip().lower().startswith("yes")


def classify_document_type_baseline(text):
    prompt = f"""
This is the start of a new document. Based on the content, classify it.

Page Content:
{text}

Choose from: Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration, Material Description, Supplier Qualification, Chain of Custody, Other.
Just respond with the type.
"""
    response = gemini_model(prompt).strip().lower().replace(".", "")
    return response.title()

In [ ]:
def run_pipeline(is_same_fn, classify_fn, label):
    results = []
    current_doc_type = None
    doc_counter = 0
    boundaries = []

    for i, page in enumerate(doc_pages):
        if i == 0:
            current_doc_type = classify_fn(page["text"])
            boundaries.append((0, "first page", current_doc_type))
        else:
            prev_text = doc_pages[i - 1]["text"]
            same = is_same_fn(prev_text, page["text"], current_doc_type)
            if not same:
                doc_counter += 1
                current_doc_type = classify_fn(page["text"])
                boundaries.append((i, "new document", current_doc_type))

        results.append({"page": i, "doc_id": doc_counter, "doc_type": current_doc_type})

    print(f"\n--- {label} ---")
    for r in results:
        print(r)
    print(f"{doc_counter + 1} documents detected across {len(results)} pages")
    return results


results_baseline = run_pipeline(
    is_same_document_baseline, classify_document_type_baseline, "BASELINE (module prompts verbatim)"
)


--- BASELINE (module prompts verbatim) ---
{'page': 0, 'doc_id': 0, 'doc_type': 'Other'}
{'page': 1, 'doc_id': 1, 'doc_type': 'Certificate Of Quality'}
{'page': 2, 'doc_id': 2, 'doc_type': 'Certificate Of Quality'}
{'page': 3, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 4, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 5, 'doc_id': 4, 'doc_type': 'Bse/Tse Declaration'}
{'page': 6, 'doc_id': 5, 'doc_type': 'Material Description'}
{'page': 7, 'doc_id': 5, 'doc_type': 'Material Description'}
{'page': 8, 'doc_id': 5, 'doc_type': 'Material Description'}
{'page': 9, 'doc_id': 5, 'doc_type': 'Material Description'}
6 documents detected across 10 pages


## Step 4: Improved logic

Two changes, both aimed at failures this specific file can produce.

**`is_same_document`** now tells the model what actually marks a boundary in these documents, and
makes the key point explicit: two pages can share a layout and a document type and still be two
different documents, if their identifiers differ. A blob of certificates for different lots is
exactly that case, and the baseline prompt gives the model no reason to look at the lot number.

**`classify_document_type`** now constrains the output to the allowed list and validates the reply
against it, instead of trusting `.title()` on free text. `.title()` also mangles `BSE/TSE` into
`Bse/Tse`, which silently breaks any downstream match on the type string.

In [ ]:
def is_same_document_v2(prev_text, curr_text, doc_type=None):
    prompt = f"""You are deciding whether two consecutive pages of a scanned pharmaceutical file
belong to the SAME document or whether the current page STARTS A NEW document.

The previous page was classified as: {doc_type or 'unknown'}

A NEW document starts when the current page shows any of:
- a new title or heading block at the top of the page
- a different document identifier (document number, certificate number, lot number, article number)
- a "Page 1 of N" style marker, or a numbering sequence that restarts

The SAME document continues when the current page shows any of:
- the word "continued", or a "Page 2 of N" style marker
- a section numbering sequence that carries on from the previous page
- a table or paragraph that is clearly cut off at the previous page break

IMPORTANT: two pages can have the SAME document type and the SAME layout and still be TWO DIFFERENT
documents. Certificates of quality for two different lots are different documents. Before answering,
compare any identifiers (lot number, document number, article number, product name) that appear on
both pages. If those identifiers differ, the answer is No.

Previous Page:
{prev_text[:2000]}

Current Page:
{curr_text[:2000]}

Respond with only: Yes or No. Do not explain."""
    response = gemini_model(prompt)
    return response.strip().lower().startswith("yes")


def classify_document_type_v2(text):
    options = ", ".join(DOC_TYPES)
    prompt = f"""This is the first page of a new document in a pharmaceutical supplier file.
Classify it into exactly one of these categories, copied character for character:

{options}

Page Content:
{text[:2000]}

Reply with the category only. No punctuation, no explanation."""
    response = gemini_model(prompt).strip().strip(".")

    # Validate against the allowed list rather than trusting .title() on free text.
    for t in DOC_TYPES:
        if response.lower() == t.lower():
            return t
    for t in DOC_TYPES:
        if t.lower() in response.lower():
            return t
    print(f"  [warn] unrecognised classification {response!r}, recording as Other")
    return "Other"


results_v2 = run_pipeline(is_same_document_v2, classify_document_type_v2, "IMPROVED")


--- IMPROVED ---
{'page': 0, 'doc_id': 0, 'doc_type': 'Cover Letter'}
{'page': 1, 'doc_id': 1, 'doc_type': 'Certificate of Quality'}
{'page': 2, 'doc_id': 2, 'doc_type': 'Certificate of Quality'}
{'page': 3, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 4, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 5, 'doc_id': 4, 'doc_type': 'BSE/TSE Declaration'}
{'page': 6, 'doc_id': 5, 'doc_type': 'Material Description'}
{'page': 7, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}
{'page': 8, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}
{'page': 9, 'doc_id': 7, 'doc_type': 'Chain of Custody'}
8 documents detected across 10 pages


## Step 5: Compare the two runs

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "page": [r["page"] for r in results_baseline],
    "baseline_doc_id": [r["doc_id"] for r in results_baseline],
    "baseline_doc_type": [r["doc_type"] for r in results_baseline],
    "improved_doc_id": [r["doc_id"] for r in results_v2],
    "improved_doc_type": [r["doc_type"] for r in results_v2],
})
comparison["differs"] = (
    (comparison.baseline_doc_id != comparison.improved_doc_id)
    | (comparison.baseline_doc_type != comparison.improved_doc_type)
)
display(comparison)

n_base = comparison.baseline_doc_id.nunique()
n_v2 = comparison.improved_doc_id.nunique()
print(f"baseline found {n_base} documents; improved found {n_v2}")
print(f"pages where the two disagree: {comparison.differs.sum()}")
print(f"total Gemini calls: {CALLS['n']}")

,page,baseline_doc_id,baseline_doc_type,improved_doc_id,improved_doc_type,differs
0,0,0,Other,0,Cover Letter,True
1,1,1,Certificate Of Quality,1,Certificate of Quality,True
2,2,2,Certificate Of Quality,2,Certificate of Quality,True
3,3,3,Packaging Specification,3,Packaging Specification,False
4,4,3,Packaging Specification,3,Packaging Specification,False
5,5,4,Bse/Tse Declaration,4,BSE/TSE Declaration,True
6,6,5,Material Description,5,Material Description,False
7,7,5,Material Description,6,Supplier Qualification,True
8,8,5,Material Description,6,Supplier Qualification,True
9,9,5,Material Description,7,Chain of Custody,True


baseline found 6 documents; improved found 8
pages where the two disagree: 7
total Gemini calls: 33


## Step 6: Final output

In [ ]:
import json

df = pd.DataFrame(results_v2)
display(df)

final_json = json.dumps(results_v2, indent=2)
with open("/content/page_metadata.json", "w") as f:
    f.write(final_json)

print("=" * 78)
print("FINAL JSON - paste this into the Google Doc")
print("=" * 78)
print(final_json)
print("=" * 78)
print("END OF JSON")

,page,doc_id,doc_type
0,0,0,Cover Letter
1,1,1,Certificate of Quality
2,2,2,Certificate of Quality
3,3,3,Packaging Specification
4,4,3,Packaging Specification
5,5,4,BSE/TSE Declaration
6,6,5,Material Description
7,7,6,Supplier Qualification
8,8,6,Supplier Qualification
9,9,7,Chain of Custody


FINAL JSON - paste this into the Google Doc
[
  {
    "page": 0,
    "doc_id": 0,
    "doc_type": "Cover Letter"
  },
  {
    "page": 1,
    "doc_id": 1,
    "doc_type": "Certificate of Quality"
  },
  {
    "page": 2,
    "doc_id": 2,
    "doc_type": "Certificate of Quality"
  },
  {
    "page": 3,
    "doc_id": 3,
    "doc_type": "Packaging Specification"
  },
  {
    "page": 4,
    "doc_id": 3,
    "doc_type": "Packaging Specification"
  },
  {
    "page": 5,
    "doc_id": 4,
    "doc_type": "BSE/TSE Declaration"
  },
  {
    "page": 6,
    "doc_id": 5,
    "doc_type": "Material Description"
  },
  {
    "page": 7,
    "doc_id": 6,
    "doc_type": "Supplier Qualification"
  },
  {
    "page": 8,
    "doc_id": 6,
    "doc_type": "Supplier Qualification"
  },
  {
    "page": 9,
    "doc_id": 7,
    "doc_type": "Chain of Custody"
  }
]
END OF JSON
